In [ ]:
import numpy as np

class NaiveBayesFromScratch:
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.parameters = {}

        for c in self.classes:
            # Filter data for only this class
            X_c = X[y == c]
            # Calculate the 'Profile' of this class
            self.parameters[c] = {
                "mean": X_c.mean(axis=0),
                "var": X_c.var(axis=0),
                "prior": X_c.shape[0] / X.shape[0] # How common is this class?
            }

    def _get_probability(self, class_val, x):
        """ The Gaussian Probability Density Function """
        mean = self.parameters[class_val]["mean"]
        var = self.parameters[class_val]["var"]
        
        # The 'Bell Curve' formula
        numerator = np.exp(- (x - mean)**2 / (2 * var))
        denominator = np.sqrt(2 * np.pi * var)
        return numerator / denominator

    def predict(self, X):
        """ Main prediction function for new data """
        predictions = []
        for x in X:
            posteriors = []
            for c in self.classes:
                # Start with the general probability of the class
                prior = np.log(self.parameters[c]["prior"])
                
                # Add the 'likelihood' of each feature (log addition = multiplication)
                probs = self._get_probability(c, x)
                # We add a tiny value to avoid log(0) errors
                likelihood = np.sum(np.log(probs + 1e-9))
                
                # Total score for this class
                posteriors.append(prior + likelihood)
            
            # Pick the class with the highest score
            predictions.append(self.classes[np.argmax(posteriors)])
        return np.array(predictions)

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# 1. Create a dummy 'Exam' dataset: 2 features (Hours Studied, Sleep), 2 classes (Pass/Fail)
X, y = make_classification(n_samples=200, n_features=2, n_redundant=0, random_state=42)

# 2. Split into Train and Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Initialize and Fit
model = NaiveBayesFromScratch()
model.fit(X_train, y_train)

# 4. Predict on UNSEEN data
test_predictions = model.predict(X_test)

# 5. Evaluate
accuracy = np.mean(test_predictions == y_test)
print(f"Accuracy on Test Set: {accuracy * 100:.2f}%")

# 6. Predict a single custom point: [1.5 hours study, 0.5 units sleep]
custom_point = np.array([[1.5, 0.5]])
res = model.predict(custom_point)
print(f"Prediction for custom point {custom_point[0]}: {'Pass' if res[0]==1 else 'Fail'}")

Accuracy on Test Set: 77.50%
Prediction for custom point [1.5 0.5]: Pass
